In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

Matplotlib is building the font cache; this may take a moment.


In [2]:
df=pd.read_csv('../data/raw/churn_data.csv.csv')

In [3]:
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,1,56,Female,68,147.58,Two year,Bank transfer,10052.03,No
1,2,69,Male,32,22.54,Month-to-month,Mailed check,686.78,No
2,3,46,Female,10,52.47,One year,Electronic check,537.88,No
3,4,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,Yes
4,5,60,Female,54,130.98,Month-to-month,Credit card,7081.28,No


In [4]:
df.shape

(100000, 9)

In [5]:
df.describe

<bound method NDFrame.describe of        CustomerID  Age  Gender  Tenure  MonthlyCharges        Contract  \
0               1   56  Female      68          147.58        Two year   
1               2   69    Male      32           22.54  Month-to-month   
2               3   46  Female      10           52.47        One year   
3               4   32    Male      22          109.67  Month-to-month   
4               5   60  Female      54          130.98  Month-to-month   
...           ...  ...     ...     ...             ...             ...   
99995       99996   31    Male      49           26.07  Month-to-month   
99996       99997   64  Female      44          123.22  Month-to-month   
99997       99998   48   Other      32           75.37  Month-to-month   
99998       99999   42  Female      60          114.00  Month-to-month   
99999      100000   72    Male      57           66.76        One year   

          PaymentMethod  TotalCharges Churn  
0         Bank transfer      10

In [6]:
df.drop('CustomerID',axis=1,inplace=True)

In [7]:
for col in df.columns:
    print(f"{col}: {df[col].isnull().sum()}")

Age: 0
Gender: 0
Tenure: 0
MonthlyCharges: 0
Contract: 0
PaymentMethod: 0
TotalCharges: 0
Churn: 0


In [8]:
df['Churn']=df['Churn'].map({'Yes':1,'No':0})

In [9]:
df.head()

,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,56,Female,68,147.58,Two year,Bank transfer,10052.03,0
1,69,Male,32,22.54,Month-to-month,Mailed check,686.78,0
2,46,Female,10,52.47,One year,Electronic check,537.88,0
3,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,1
4,60,Female,54,130.98,Month-to-month,Credit card,7081.28,0


In [10]:
# Unnecessary ID column ko drop kar dete hain
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)

# Sabhi Categorical Columns ke Unique Values dekhte hain
categorical_cols = df.select_dtypes(include=['object']).columns

print("--- CATEGORICAL COLUMNS & UNIQUE VALUES ---\n")
for col in categorical_cols:
    print(f"📌 Column: '{col}'")
    print(df[col].value_counts())
    print("-" * 40)

--- CATEGORICAL COLUMNS & UNIQUE VALUES ---

📌 Column: 'Gender'
Gender
Female    48256
Male      47787
Other      3957
Name: count, dtype: int64
----------------------------------------
📌 Column: 'Contract'
Contract
Month-to-month    54915
One year          25261
Two year          19824
Name: count, dtype: int64
----------------------------------------
📌 Column: 'PaymentMethod'
PaymentMethod
Electronic check    34892
Mailed check        25221
Credit card         20032
Bank transfer       19855
Name: count, dtype: int64
----------------------------------------


C:\Users\ks868\AppData\Local\Temp\ipykernel_12640\614229491.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


In [11]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce')

In [12]:
df['TotalCharges'].isnull().sum()

np.int64(0)

In [13]:
df_encoded=pd.get_dummies(df, drop_first=True,dtype=int)

In [14]:
df_encoded.head()

,Age,Tenure,MonthlyCharges,TotalCharges,Churn,Gender_Male,Gender_Other,Contract_One year,Contract_Two year,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,56,68,147.58,10052.03,0,0,0,0,1,0,0,0
1,69,32,22.54,686.78,0,1,0,0,0,0,0,1
2,46,10,52.47,537.88,0,0,0,1,0,0,1,0
3,32,22,109.67,2390.04,1,1,0,0,0,0,0,1
4,60,54,130.98,7081.28,0,0,0,0,0,1,0,0


In [15]:
df_encoded.corr()['Churn'].sort_values(ascending=False)

Churn                             1.000000
MonthlyCharges                    0.250055
TotalCharges                      0.023860
Gender_Male                       0.004911
Gender_Other                      0.002996
PaymentMethod_Mailed check        0.002336
PaymentMethod_Electronic check    0.002246
PaymentMethod_Credit card        -0.003737
Age                              -0.005875
Contract_Two year                -0.171817
Tenure                           -0.190924
Contract_One year                -0.202482
Name: Churn, dtype: float64

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [17]:
X=df_encoded.drop('Churn', axis=1)
y=df_encoded['Churn']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=45, stratify=y)

In [19]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scalled=scaler.fit_transform(X_train)
x_test_scalled=scaler.transform(X_test)

In [20]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_scalled, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [21]:
y_pred=rf_model.predict(x_test_scalled)

In [22]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [23]:
classification_report=classification_report(y_test,y_pred)
confusion_matrix=confusion_matrix(y_test,y_pred)
Accuracy=accuracy_score(y_test,y_pred)

In [24]:
print("Classification Report:\n", classification_report)
print("Confusion Matrix:\n", confusion_matrix)  
print("Accuracy:", Accuracy)

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.85      0.81     13371
           1       0.63      0.50      0.56      6629

    accuracy                           0.74     20000
   macro avg       0.70      0.68      0.68     20000
weighted avg       0.73      0.74      0.73     20000

Confusion Matrix:
 [[11423  1948]
 [ 3322  3307]]
Accuracy: 0.7365


In [25]:
from imblearn.over_sampling import SMOTE

In [26]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train_res, y_train_res)

y_pred_xgb = xgb_model.predict(X_test_scaled)

print(f"🚀 Improved Model Accuracy: {round(accuracy_score(y_test, y_pred_xgb) * 100, 2)}%\n")
print("New Classification Report:\n", classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))




🚀 Improved Model Accuracy: 75.72%

New Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.84      0.82     13371
           1       0.64      0.60      0.62      6629

    accuracy                           0.76     20000
   macro avg       0.73      0.72      0.72     20000
weighted avg       0.75      0.76      0.75     20000

Confusion Matrix:
 [[11192  2179]
 [ 2678  3951]]


In [27]:
import sys
print(sys.executable)


c:\Users\ks868\OneDrive\krishna24\CustomerSaver-AI\venv\Scripts\python.exe


In [28]:
import sys
!{sys.executable} -m pip install xgboost


In [30]:
import joblib 
joblib.dump(xgb_model, '../models/xgb_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X.columns.tolist(), '../models/feature_columns.pkl')


FileNotFoundError: [Errno 2] No such file or directory: '../models/xgb_model.pkl'